In [ ]:
#Installing Packages
# !pip install dash

In [ ]:
# !pip install --upgrade typing-extensions
# !pip install --upgrade pydantic
# !pip install pydantic==1.10.13

In [1]:
#Importing the Libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html
import os

In [8]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html
import os
import plotly.graph_objects as go 

# --- Global Constants ---
COLOR_MAP = {'Technology': '#007acc', 'Office Supplies': '#ff8c00', 'Furniture': '#2e8b57'}
REGION_COLORS = ['#007acc', '#ff8c00', '#2e8b57', '#9370db'] 
FORECAST_COLOR = '#9370db' 
SHIP_MODE_COLORS = REGION_COLORS # Used for Ship Mode and now Regions

# Define fixed heights for consistent layout
ROW_1_HEIGHT = '400px' # Height for the full-width line chart block (Chart 1)
ROW_HEIGHT = ROW_1_HEIGHT # Set height for all other row blocks equal to Chart 1's height
CHART_PADDING = 20 # 10px top + 10px bottom padding in the internal chart container
CHART_INNER_HEIGHT = f"{int(ROW_HEIGHT.replace('px', '')) - CHART_PADDING}px" # Calculated to be 380px

# --- Chart Explanations and Recommendations ---
EXPLANATIONS = {
    'monthly-sales-trend': {
        'title': '1. Total Monthly Sales Trend',
        'explanation': 'This line chart visualizes the total sales aggregated by month over the entire dataset period (2014-2017). It highlights clear seasonality, showing annual peaks in Q4.',
        'recommendation': 'Analyze the successful strategies (e.g., promotions, inventory build-up) employed during the end-of-year peak and look for opportunities to smooth out the cyclical sales trough in the beginning of the year.'
    },
    'sales-by-category': {
        'title': '2. Total Sales by Product Category',
        'explanation': 'This bar chart shows the total revenue generated by the three primary product categories: Technology, Office Supplies, and Furniture.',
        'recommendation': 'The **Technology** category is the clear primary revenue driver. Focus promotional efforts and inventory management on the top-performing category to maximize overall profit margins.'
    },
    'sales-by-state': {
        'title': '3. Top 10 States by Total Sales',
        'explanation': 'This chart ranks the top 10 US states contributing the most to overall sales revenue, highlighting key geographical markets.',
        'recommendation': 'Leverage the high performance in top states (e.g., California, New York) by deploying specialized sales teams and tailored marketing campaigns to protect and grow these core markets.'
    },
    'sales-by-segment-donut': {
        'title': '4. Sales Distribution by Customer Segment',
        'explanation': 'This donut chart breaks down total sales revenue across the three customer segments: Consumer, Corporate, and Home Office.',
        'recommendation': 'The **Consumer** segment is dominant. Develop loyalty programs for this group while creating targeted value propositions to increase the revenue share from the Corporate and Home Office segments.'
    },
    'sales-by-ship-mode-boxplot': {
        'title': '5. Sales Distribution by Ship Mode',
        'explanation': 'This box plot compares the distribution of individual order sales values across different shipping modes. It indicates the typical value range for each delivery speed.',
        'recommendation': 'Orders shipped via **Same Day** show a higher average sale value. Ensure premium shipping options are efficiently managed and priced appropriately to cater to these high-value transactions.'
    },
    'sales-by-category-subcategory-treemap': {
        'title': '6. Sales Breakdown by Category and Sub-Category',
        'explanation': 'This treemap provides a detailed, hierarchical view of sales, showing how each Sub-Category contributes to its parent Category’s total sales.',
        'recommendation': 'Identify high-value, high-volume Sub-Categories (e.g., Phones, Binders) to prioritize inventory. Use this view to spot underperforming sub-categories that may need product retirement or marketing intervention.'
    },
    'sales-forecast-chart': {
        'title': '7. Time Series Sales Forecast (with Uncertainty)',
        'explanation': 'This chart displays historical sales and a short-term future forecast. The shaded area represents the 95% confidence interval (the likely range of future outcomes).',
        'recommendation': 'Use the forecast’s expected value (`yhat`) to guide initial procurement and staffing levels. Plan for the high end of the confidence interval (`yhat_upper`) to mitigate stock-out risks during peak sales periods.'
    },
    'sales-composition-by-region': { # New Chart 8
        'title': '8. Monthly Sales Composition by Region (100% Stacked)',
        'explanation': 'This 100% stacked area chart visualizes the proportional sales contribution of each Region (West, East, Central, South) over time. This view clearly shows shifts in market dominance between regions.',
        'recommendation': 'The **West** and **East** regions consistently form the largest portion of sales. Focus resource allocation and growth initiatives on maintaining their leading market share, while analyzing the Central and South regions for targeted expansion opportunities.'
    }
}

# --- 1. Data Loading and Preprocessing ---
try:
    # Assuming 'superstore_final_dataset.csv' is in the same directory
    df = pd.read_csv('data/superstore_final_dataset.csv', encoding='latin1')
    
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], infer_datetime_format=True, dayfirst=True)
    df = df.sort_values('Order_Date')
    
except Exception as e:
    print(f"Error loading or processing data: {e}")
    # Create an empty DataFrame to prevent app crash
    df = pd.DataFrame({'Order_Date': [], 'Sales': [], 'Category': [], 'Sub_Category': [], 'Segment': [], 'State': [], 'Ship_Mode': []})


# --- 2. Key Data Aggregations ---
monthly_sales = df.set_index('Order_Date').resample('M')['Sales'].sum().reset_index()
monthly_sales.columns = ['Order_Date', 'Total Sales']

historical_data_for_forecast = monthly_sales[monthly_sales['Order_Date'].dt.year <= 2017].rename(columns={'Order_Date': 'ds', 'Total Sales': 'y'})
forecast_dates = pd.to_datetime(['2018-01-31', '2018-02-28', '2018-03-31'])
forecast_data = pd.DataFrame({
    'ds': forecast_dates, 
    'yhat': [40000, 42000, 50000],  
    'yhat_lower': [35000, 37000, 44000], 
    'yhat_upper': [45000, 47000, 56000],
})
historical_and_forecast = historical_data_for_forecast.rename(columns={'y':'yhat'})

sales_by_category = df.groupby('Category')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)
sales_by_state = df.groupby('State')['Sales'].sum().nlargest(10).reset_index().sort_values('Sales', ascending=True)
sales_by_segment = df.groupby('Segment')['Sales'].sum().reset_index()
sales_by_ship_mode = df[['Ship_Mode', 'Sales']]
sales_by_sub_category = df.groupby(['Category', 'Sub_Category'])['Sales'].sum().reset_index()

# NEW DATA AGGREGATION for Chart 8: Monthly Sales by Region
df_monthly_region = df.set_index('Order_Date').groupby('Region').resample('M')['Sales'].sum().reset_index()
df_monthly_region.rename(columns={'Order_Date': 'Order_Date', 'Sales': 'Sales'}, inplace=True)


# --- 3. Visualization Functions (Unchanged) ---
def create_monthly_sales_chart(df_monthly):
    """Line Chart: Total Monthly Sales Trend"""
    fig = px.line(df_monthly, x='Order_Date', y='Total Sales', title='1. Total Monthly Sales Trend', template='plotly_white')
    fig.update_traces(mode='lines+markers', line=dict(color=COLOR_MAP['Technology']), marker=dict(size=4))
    fig.update_layout(title_x=0.5, yaxis_title="Sales ($)")
    return fig

def create_category_sales_chart(df_category):
    """Bar Chart: Total Sales by Product Category"""
    fig = px.bar(df_category, x='Category', y='Sales', title='2. Total Sales by Product Category', template='plotly_white', color='Category', color_discrete_map=COLOR_MAP)
    fig.update_layout(title_x=0.5, showlegend=False, xaxis_title="", yaxis_title="Sales ($)")
    return fig

def create_state_sales_chart(df_state):
    """Horizontal Bar Chart: Top 10 States by Total Sales"""
    fig = px.bar(df_state, x='Sales', y='State', orientation='h', title='3. Top 10 States by Total Sales', template='plotly_white')
    fig.update_traces(marker_color=COLOR_MAP['Furniture'])
    fig.update_layout(title_x=0.5, yaxis={'categoryorder':'total ascending'}, xaxis_title="Sales ($)", yaxis_title="")
    return fig

def create_segment_donut_chart(df_segment):
    """Donut Chart: Sales Distribution by Customer Segment"""
    fig = px.pie(df_segment, values='Sales', names='Segment', hole=0.4, title='4. Sales Distribution by Customer Segment', template='plotly_white', color='Segment', color_discrete_map={'Consumer': SHIP_MODE_COLORS[0], 'Corporate': SHIP_MODE_COLORS[1], 'Home Office': SHIP_MODE_COLORS[2]})
    fig.update_traces(textposition='inside', textinfo='percent+label', marker=dict(line=dict(color='#000000', width=1)))
    fig.update_layout(title_x=0.5, showlegend=False)
    return fig

def create_ship_mode_boxplot(df_ship_mode):
    """Box Plot: Sales Distribution by Ship Mode"""
    fig = px.box(df_ship_mode, x='Ship_Mode', y='Sales', title='5. Sales Distribution by Ship Mode', template='plotly_white', color='Ship_Mode', color_discrete_sequence=SHIP_MODE_COLORS)
    fig.update_layout(title_x=0.5, showlegend=False, xaxis_title="", yaxis_title="Sales ($)")
    return fig

def create_category_subcategory_treemap(df_sub_category):
    """Treemap: Sales Breakdown by Category and Sub-Category"""
    fig = px.treemap(df_sub_category, path=[px.Constant("All"), 'Category', 'Sub_Category'], values='Sales', color='Category', color_discrete_map={'All': 'lightgrey', **COLOR_MAP}, title='6. Sales Breakdown by Category and Sub-Category', template='plotly_white')
    fig.update_layout(title_x=0.5, margin = dict(t=50, l=10, r=10, b=10))
    return fig

def create_forecast_chart(df_forecast, df_historical):
    """Line Chart with Forecasted Confidence Interval"""
    
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_historical['ds'],
        y=df_historical['yhat'],
        mode='lines+markers',
        name='Historical Sales',
        line=dict(color=COLOR_MAP['Technology'], width=2),
        marker=dict(size=4)
    ))

    forecast_plot_data = pd.concat([df_historical.tail(1), df_forecast], ignore_index=True)
    
    fig.add_trace(go.Scatter(
        x=forecast_plot_data['ds'],
        y=forecast_plot_data['yhat'],
        mode='lines',
        name='Forecasted Sales',
        line=dict(color=FORECAST_COLOR, width=2, dash='dash')
    ))
    
    fig.add_trace(go.Scatter(
        x=df_forecast['ds'],
        y=df_forecast['yhat_upper'],
        mode='lines',
        line=dict(width=0),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=df_forecast['ds'],
        y=df_forecast['yhat_lower'],
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(147, 112, 219, 0.2)',
        name='95% Confidence Interval'
    ))

    fig.update_layout(
        title='7. Time Series Sales Forecast (with Uncertainty)',
        title_x=0.5,
        template='plotly_white',
        yaxis_title="Sales ($)",
        xaxis_title="Date",
        hovermode="x unified"
    )
    return fig


# --- 4. Initialize Dash App ---
app = Dash(__name__)
server = app.server

# --- Custom Insight Block Function ---
def create_insight_block(chart_id, height):
    """Generates the HTML block for chart explanations and recommendations."""
    insight = EXPLANATIONS[chart_id]
    return html.Div(
        style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '15px', 'height': height, 'boxShadow': '2px 2px 5px #aaaaaa', 'borderLeft': f'3px solid {COLOR_MAP["Technology"]}', 'overflowY': 'auto'},
        children=[
            html.H4("Chart Summary", style={'color': '#1f4a66', 'borderBottom': '1px solid #eee', 'paddingBottom': '5px', 'marginBottom': '10px'}),
            html.P(insight['explanation'], style={'fontSize': '14px', 'lineHeight': '1.5'}),
            
            html.H4("Key Recommendation", style={'color': '#2e8b57', 'marginTop': '15px', 'borderBottom': '1px solid #eee', 'paddingBottom': '5px', 'marginBottom': '10px'}),
            html.P(insight['recommendation'], style={'fontSize': '14px', 'lineHeight': '1.5'})
        ]
    )

def create_region_composition_chart(df_region_monthly):
    """Stacked Area Chart: Monthly Sales Composition by Region (100% Stacked)"""
    fig = px.area(
        df_region_monthly, 
        x="Order_Date", 
        y="Sales", 
        color="Region",
        title="8. Monthly Sales Composition by Region (100% Stacked)",
        template='plotly_white',
        groupnorm='percent', # Key for 100% stacked area
        color_discrete_sequence=REGION_COLORS 
    )
    
    fig.update_layout(
        title_x=0.5, 
        yaxis_title="Sales Proportion (%)", 
        hovermode="x unified",
        yaxis_tickformat='.0%', # Format y-axis as percentage
        legend_title="Region"
    )
    return fig


# --- 5. Define the Layout ---

app.layout = html.Div(style={'backgroundColor': '#f0f2f5', 'padding': '20px'}, children=[
    
    # Dashboard Header
    html.H1(
        children='Superstore Sales & Forecasting Dashboard',
        style={'textAlign': 'center', 'color': '#1f4a66', 'marginBottom': '30px', 'padding': '10px 0', 'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 10px #aaaaaa'}
    ),
    
    # --- Row 1: Chart 1 (Monthly Sales Trend) ---
    # Chart (Full Width)
    html.Div(className='row', style={'padding': '10px', 'backgroundColor': 'white', 'borderRadius': '8px', 'marginBottom': '10px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_1_HEIGHT}, children=[
        dcc.Graph(id='monthly-sales-trend', figure=create_monthly_sales_chart(monthly_sales), style={'height': ROW_1_HEIGHT, 'padding': '10px'})
    ]),
    # Explanation (Full Width, right below the chart)
    html.Div(className='row', style={'marginBottom': '20px'}, children=[
        html.Div(style={'width': '100%', 'padding': '10px'}, children=[
            create_insight_block('monthly-sales-trend', 'auto') 
        ])
    ]),
    
    # --- Row 2: Charts 2 & 3 ---
    html.Div(className='row', style={'display': 'flex', 'flex-wrap': 'wrap', 'marginBottom': '20px'}, children=[
        # Chart 2 (Sales by Category) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingRight': '5px'}, children=[ 
                 html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-by-category', figure=create_category_sales_chart(sales_by_category), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-by-category', '370px')
            ])
        ]),
        
        # Chart 3 (Sales by State) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingRight': '5px'}, children=[ 
                html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-by-state', figure=create_state_sales_chart(sales_by_state), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-by-state', '370px')
            ])
        ])
    ]),
    
    # --- Row 3: Charts 4 & 5 ---
    html.Div(className='row', style={'display': 'flex', 'flex-wrap': 'wrap', 'marginBottom': '20px'}, children=[
        # Chart 4 (Sales by Segment) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingRight': '5px'}, children=[ 
                html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-by-segment-donut', figure=create_segment_donut_chart(sales_by_segment), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-by-segment-donut', '370px')
            ])
        ]),
        
        # Chart 5 (Sales by Ship Mode) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingRight': '5px'}, children=[ 
                html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-by-ship-mode-boxplot', figure=create_ship_mode_boxplot(sales_by_ship_mode), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-by-ship-mode-boxplot', '370px')
            ])
        ])
    ]),
    
    # --- Row 4: Charts 6 & 7 ---
    html.Div(className='row', style={'display': 'flex', 'flex-wrap': 'wrap', 'marginBottom': '20px'}, children=[
        # Chart 6 (Treemap) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingRight': '5px'}, children=[ 
                 html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-by-category-subcategory-treemap', figure=create_category_subcategory_treemap(sales_by_sub_category), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '50%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-by-category-subcategory-treemap', '370px')
            ])
        ]),
        
        # Chart 8 (New Stacked Area Chart) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '75%', 'paddingRight': '5px'}, children=[ 
                html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-composition-by-region', figure=create_region_composition_chart(df_monthly_region), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '25%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-composition-by-region', '370px')
            ])
        ]),
        
        # Chart 7 (Forecast) + Text (50% of the row)
        html.Div(style={'width': '100%', 'padding': '10px', 'display': 'flex'}, children=[
            # Chart (50% of this 50% block)
            html.Div(style={'width': '75%', 'paddingRight': '5px'}, children=[ 
                html.Div(style={'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '2px 2px 5px #aaaaaa', 'height': ROW_HEIGHT}, children=[
                    dcc.Graph(id='sales-forecast-chart', figure=create_forecast_chart(forecast_data, historical_and_forecast), style={'height': CHART_INNER_HEIGHT, 'padding': '10px'})
                ])
            ]),
            # Text Insight (50% of this 50% block)
            html.Div(style={'width': '25%', 'paddingLeft': '5px'}, children=[ 
                create_insight_block('sales-forecast-chart', '370px')
            ])
        ])
    ]),
    
    # Footer
    html.Div(style={'textAlign': 'center', 'padding': '20px', 'color': '#5a5a5a'}, children=[
        html.P("Dashboard developed using Dash and Plotly for Superstore Sales Analysis and Forecasting.")
    ])
])


# --- 6. Run the App ---
if __name__ == '__main__':
    app.run(debug=True, host='127.0.0.1', port=8050)